In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

In [3]:
df = pd.read_csv('Apart Deal_6.csv')
df.head()

,지역코드,시군구,법정동,지번,아파트,거래일,건축년도,층,전용면적,거래금액,기준금리,위도,경도,인근학교수,인근역수,세대수,브랜드,브랜드여부
0,11110,서울특별시 종로구,견지동,110,대성스카이렉스,2015-02-04,2008.0,12,137.55,77000,2.00,37.57243,126.982527,4,5,72928.0,기타,0
1,11110,서울특별시 종로구,견지동,110,대성스카이렉스,2015-05-25,2008.0,7,149.72,90000,1.75,37.57243,126.982527,4,5,73032.0,기타,0
2,11110,서울특별시 종로구,견지동,110,대성스카이렉스,2015-05-29,2008.0,8,116.03,75000,1.75,37.57243,126.982527,4,5,73032.0,기타,0
3,11110,서울특별시 종로구,견지동,110,대성스카이렉스,2015-08-20,2008.0,5,116.03,74000,1.50,37.57243,126.982527,4,5,72691.0,기타,0
4,11110,서울특별시 종로구,견지동,110,대성스카이렉스,2015-10-21,2008.0,15,149.80,102000,1.50,37.57243,126.982527,4,5,72890.0,기타,0


In [5]:
len(df["브랜드"].unique())

61

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

cols = ['거래금액', '전용면적', '층', '건축년도', '지역코드', '기준금리', '인근학교수', '인근역수', '세대수', '브랜드여부']
data = df[cols].copy()

# 숫자형 컬럼 정리
num_cols = ['거래금액', '전용면적', '층', '건축년도', '기준금리', '인근학교수', '인근역수', '세대수']

for col in num_cols:
    data[col] = (
        data[col]
        .astype(str)
        .str.strip()
        .str.replace(',', '', regex=False)
    )
    data[col] = pd.to_numeric(data[col], errors='coerce')

# 범주형 컬럼 정리
data['지역코드'] = data['지역코드'].astype(str).str.strip()
data['브랜드'] = data['브랜드'].astype(str).str.strip()

# 건물연식 생성
data['건물연식'] = 2026 - data['건축년도']

# 결측치 제거
data = data.dropna(subset=['거래금액', '전용면적', '층', '건물연식', '지역코드', '기준금리', '인근학교수', '인근역수', '세대수', '브랜드여부'])

# 피처 / 타깃
X = data[['전용면적', '층', '건물연식', '지역코드', '기준금리', '인근학교수', '인근역수', '세대수', '브랜드여부']]
y = data['거래금액']

# 원-핫 인코딩
X = pd.get_dummies(X, columns=['지역코드'], drop_first=True)

# 데이터가 크면 우선 샘플링 추천
sample_idx = X.sample(n=100000, random_state=42).index

X_sample = X.loc[sample_idx]
y_sample = y.loc[sample_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    random_state=42
)

model = HistGradientBoostingRegressor(
    max_iter=100,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))

# 처음 데이터               MAE: 7662, R2 Score : 0.761
# 기준금리 추가시           MAE: 7381, R2 Score : 0.788 
# 인근학교, 역 수 추가시    MAE: 7104, R2 Score : 0.805
# 세대수 추가시             MAE: 6508, R2 Score : 0.833
# 브랜드여부 추가시         MAE: 6427, R2 Score : 0.835

KeyError: '브랜드'

In [14]:
from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt

# 중요도 계산용 샘플
X_imp = X_test.sample(n=min(10000, len(X_test)), random_state=42)
y_imp = y_test.loc[X_imp.index]

# permutation importance 계산
result = permutation_importance(
    model,
    X_imp,
    y_imp,
    n_repeats=5,
    random_state=42,
    scoring='r2',
    n_jobs=1
)

# 개별 중요도 생성
importance_df = pd.DataFrame({
    'feature': X_imp.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
})

# 지역코드 원핫 컬럼들을 하나의 피처명으로 합치기
importance_df['feature_group'] = importance_df['feature'].apply(
    lambda x: '지역코드' if x.startswith('지역코드_') else x
)

# 합친 피처 중요도만 계산
group_importance = (
    importance_df
    .groupby('feature_group', as_index=False)
    .agg(
        importance_mean=('importance_mean', 'sum'),
        importance_std=('importance_std', 'mean')
    )
    .sort_values('importance_mean', ascending=False)
)

# 합친 결과만 출력
print(group_importance)

  feature_group  importance_mean  importance_std
6          지역코드         0.557844        0.000161
5          전용면적         0.388623        0.010640
0          건물연식         0.167162        0.006296
2           세대수         0.145177        0.007366
3          인근역수         0.073968        0.006224
1          기준금리         0.026185        0.001564
4         인근학교수         0.022857        0.001119
7             층         0.012070        0.000658
